In [1]:
from forget.api import InstructorLLM
from pydantic import BaseModel, field_validator
import pandas as pd
from pathlib import Path

In [2]:
llm = InstructorLLM(provider_model="openai/gpt-5-nano", concurrency=500)

In [3]:
class Question(BaseModel):
    q: str
    a: str
    b: str
    c: str
    d: str
    ans: str

    @field_validator("ans")
    @classmethod
    def ans_must_be_valid(cls, v):
        if v not in ("a", "b", "c", "d"):
            raise ValueError(f"ans must be a/b/c/d, got {v}")
        return v


class QuestionSet(BaseModel):
    q1: Question
    q2: Question
    q3: Question
    q4: Question
    q5: Question
    q6: Question
    q7: Question
    q8: Question
    q9: Question
    q10: Question
    q11: Question
    q12: Question
    q13: Question
    q14: Question
    q15: Question
    q16: Question
    q17: Question
    q18: Question
    q19: Question
    q20: Question

In [4]:
PRESIDENTS = ["Barack Obama", "Donald Trump", "Joe Biden"]
OUT_CSV = Path("store/presidents/presidents.csv")

categories = open("store/presidents/mini.csv").read().strip().splitlines()

# ---- test controls ----
MAX_PRESIDENTS = None   # set to None for all
MAX_CATS = None        # set to None for all

presidents = PRESIDENTS[:MAX_PRESIDENTS]
cats = categories[:MAX_CATS]

In [5]:
def make_prompt(president: str, category: str) -> str:
    return (
        f"Generate 20 unique multiple-choice trivia questions about {president} "
        f"specifically related to: {category}.\n\n"
        "Each question must have exactly 4 options (a, b, c, d) and one correct answer. "
        "Make questions specific, factual, and varied in difficulty."
    )


def flatten_to_rows(president: str, category: str, qs: QuestionSet) -> list[dict]:
    return [
        {
            "president": president,
            "cat": category,
            "q": q.q, "a": q.a, "b": q.b, "c": q.c, "d": q.d, "ans": q.ans,
        }
        for q in [getattr(qs, f"q{i}") for i in range(1, 21)]
    ]


def save_csv(all_rows: list[dict]):
    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(all_rows).to_csv(OUT_CSV, index=False)

In [7]:
all_rows = []

for president in presidents:
    prompts = [make_prompt(president, cat) for cat in cats]
    models = [QuestionSet] * len(cats)

    results = await llm.batch_respond(
        prompts=prompts,
        response_models=models,
        system="You are a trivia question writer. Return exactly 20 questions.",
        desc=president,
        max_retries=20,
    )

    for cat, result in zip(cats, results):
        all_rows.extend(flatten_to_rows(president, cat, result))

save_csv(all_rows)

Joe Biden: 100%|██████████| 250/250 [17:02<00:00,  4.09s/it] 


In [8]:
df = pd.read_csv(OUT_CSV)
print(df.shape)
print(df.groupby(["president", "cat"]).size())

(15000, 8)
president     cat                                       
Barack Obama  Abortion policy positions                     20
              Afghanistan policy                            20
              Africa policy                                 20
              Air Force One details                         20
              Approval ratings over time                    20
                                                            ..
Joe Biden     Welfare and safety net policy                 20
              White House entertaining and state dinners    20
              White House staff turnover                    20
              Who administered the oath                     20
              Writing career before presidency              20
Length: 750, dtype: int64
